In [1]:
# ============================================================
# EX4: INFORMATION RETRIEVAL USING NLTK
# Amazon Fine Food Reviews
# ============================================================


# ============================================================
# 1. INSTALL / IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import string

import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 2. DOWNLOAD REQUIRED NLTK RESOURCES
# ============================================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("NLTK resources downloaded successfully!")


# ============================================================
# 3. LOAD AMAZON FINE FOOD REVIEWS DATASET
# ============================================================

file_path = r"Downloads\Reviews.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")

print("\nDataset shape:")
print(df.shape)

print("\nFirst 5 records:")
print(df.head())


# ============================================================
# 4. CHECK COLUMN NAMES
# ============================================================

print("\nColumn names:")
print(df.columns.tolist())


# ============================================================
# 5. SELECT TEXT COLUMN
# ============================================================

reviews = df[["Text"]].copy()

print("\nTotal reviews:", len(reviews))


# ============================================================
# 6. REMOVE MISSING / NULL REVIEWS
# ============================================================

reviews = reviews.dropna(
    subset=["Text"]
)

print(
    "\nReviews after removing missing values:",
    len(reviews)
)


# ============================================================
# 7. REMOVE DUPLICATE REVIEWS
# ============================================================

reviews = reviews.drop_duplicates(
    subset=["Text"]
)

print(
    "Reviews after removing duplicates:",
    len(reviews)
)


# ============================================================
# 8. RETAIN FIRST 10,000 REVIEWS
# ============================================================

reviews = reviews.head(10000).copy()

print(
    "Reviews selected for processing:",
    len(reviews)
)


# ============================================================
# 9. CREATE STOPWORDS SET
# ============================================================

stop_words = set(
    stopwords.words("english")
)

print(
    "\nNumber of English stopwords:",
    len(stop_words)
)


# ============================================================
# 10. TEXT PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(
        r"\d+",
        "",
        text
    )

    # Remove punctuation and special characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        "",
        text
    )

    # Tokenize text
    tokens = word_tokenize(text)

    # Remove stopwords
    cleaned_tokens = []

    for word in tokens:

        if word not in stop_words:
            cleaned_tokens.append(word)

    # Join cleaned tokens
    cleaned_text = " ".join(
        cleaned_tokens
    )

    return cleaned_text


# ============================================================
# 11. APPLY PREPROCESSING
# ============================================================

reviews["Cleaned_Text"] = reviews["Text"].apply(
    preprocess_text
)

print("\nText preprocessing completed!")


# ============================================================
# 12. DISPLAY ORIGINAL AND CLEANED REVIEWS
# ============================================================

print("\n========================================")
print("ORIGINAL VS CLEANED REVIEWS")
print("========================================")

for i in range(5):

    print("\nReview", i + 1)

    print("Original:")
    print(reviews.iloc[i]["Text"])

    print("\nCleaned:")
    print(reviews.iloc[i]["Cleaned_Text"])

    print("----------------------------------------")


# ============================================================
# 13. REMOVE EMPTY CLEANED REVIEWS
# ============================================================

reviews = reviews[
    reviews["Cleaned_Text"].str.strip() != ""
].copy()

print(
    "\nReviews available after preprocessing:",
    len(reviews)
)


# ============================================================
# 14. TF-IDF VECTORIZATION
# ============================================================

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(
    reviews["Cleaned_Text"]
)


# ============================================================
# 15. DISPLAY TF-IDF MATRIX SHAPE
# ============================================================

print("\n========================================")
print("TF-IDF INFORMATION")
print("========================================")

print(
    "TF-IDF Matrix Shape:",
    tfidf_matrix.shape
)


# ============================================================
# 16. DISPLAY VOCABULARY SIZE
# ============================================================

vocabulary_size = len(
    tfidf_vectorizer.vocabulary_
)

print(
    "Total number of unique terms:",
    vocabulary_size
)


# ============================================================
# 17. DISPLAY SOME VOCABULARY TERMS
# ============================================================

print("\nFirst 20 vocabulary terms:")

vocabulary = list(
    tfidf_vectorizer.vocabulary_.keys()
)

print(
    vocabulary[:20]
)


# ============================================================
# 18. INFORMATION RETRIEVAL FUNCTION
# ============================================================

def retrieve_reviews(query, top_n=5):

    # --------------------------------------------------------
    # Preprocess the query
    # --------------------------------------------------------

    cleaned_query = preprocess_text(
        query
    )

    print("\n========================================")
    print("SEARCH QUERY")
    print("========================================")

    print("Original Query:", query)

    print(
        "Cleaned Query:",
        cleaned_query
    )

    # --------------------------------------------------------
    # Convert query into TF-IDF vector
    # --------------------------------------------------------

    query_vector = tfidf_vectorizer.transform(
        [cleaned_query]
    )

    # --------------------------------------------------------
    # Calculate cosine similarity
    # --------------------------------------------------------

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # --------------------------------------------------------
    # Get top 5 similar reviews
    # --------------------------------------------------------

    top_indices = similarity_scores.argsort()[
        -top_n:
    ][::-1]

    # --------------------------------------------------------
    # Create result dataframe
    # --------------------------------------------------------

    results = reviews.iloc[
        top_indices
    ][
        ["Text", "Cleaned_Text"]
    ].copy()

    results["Similarity_Score"] = (
        similarity_scores[top_indices]
    )

    # Reorder columns
    results = results[
        [
            "Similarity_Score",
            "Text",
            "Cleaned_Text"
        ]
    ]

    return results


# ============================================================
# 19. TEST QUERY 1
# ============================================================

query1 = "great product with fast shipping"

result1 = retrieve_reviews(
    query1,
    top_n=5
)

print("\nTop 5 Relevant Reviews:")
print(result1.to_string(index=False))


# ============================================================
# 20. TEST QUERY 2
# ============================================================

query2 = "disappointed"

result2 = retrieve_reviews(
    query2,
    top_n=5
)

print("\nTop 5 Relevant Reviews:")
print(result2.to_string(index=False))


# ============================================================
# 21. TEST QUERY 3
# ============================================================

query3 = "excellent taste"

result3 = retrieve_reviews(
    query3,
    top_n=5
)

print("\nTop 5 Relevant Reviews:")
print(result3.to_string(index=False))


# ============================================================
# 22. TEST QUERY 4
# ============================================================

query4 = "poor packaging"

result4 = retrieve_reviews(
    query4,
    top_n=5
)

print("\nTop 5 Relevant Reviews:")
print(result4.to_string(index=False))


# ============================================================
# 23. QUERY COMPARISON
# ============================================================

print("\n\n========================================")
print("QUERY EVALUATION")
print("========================================")

queries = [
    query1,
    query2,
    query3,
    query4
]

results_list = [
    result1,
    result2,
    result3,
    result4
]

for query, result in zip(
    queries,
    results_list
):

    print("\nQuery:", query)

    print(
        "Highest Similarity Score:",
        round(
            result["Similarity_Score"].iloc[0],
            4
        )
    )

    print(
        "Top Retrieved Review:"
    )

    print(
        result["Text"].iloc[0]
    )

    print("----------------------------------------")


# ============================================================
# 24. DISPLAY TOP 5 RESULTS IN A CLEAN FORMAT
# ============================================================

def display_results(query, results):

    print("\n\n================================================")
    print("QUERY:", query)
    print("================================================")

    for i, row in results.iterrows():

        print(
            "\nSimilarity Score:",
            round(
                row["Similarity_Score"],
                4
            )
        )

        print(
            "Original Review:"
        )

        print(
            row["Text"]
        )

        print(
            "\nCleaned Review:"
        )

        print(
            row["Cleaned_Text"]
        )

        print(
            "------------------------------------------------"
        )


# Display all query results

display_results(
    query1,
    result1
)

display_results(
    query2,
    result2
)

display_results(
    query3,
    result3
)

display_results(
    query4,
    result4
)


# ============================================================
# 25. FINAL OBSERVATION
# ============================================================

print("\n\n========================================")
print("FINAL OBSERVATION")
print("========================================")

print("""
The Information Retrieval system successfully retrieves
the most relevant customer reviews for a given search query.

TF-IDF converts the reviews and search query into numerical
vectors based on the importance of words.

Cosine Similarity measures the similarity between the query
and each review.

A higher similarity score indicates that the review is more
relevant to the user's search query.

Different queries retrieve different reviews based on the
words present in the query.
""")


# ============================================================
# 26. BUSINESS APPLICATION
# ============================================================

print("\n========================================")
print("BUSINESS APPLICATION")
print("========================================")

print("""
1. Customer support executives can quickly find reviews
   related to a customer's complaint.

2. The company can identify common issues such as poor
   packaging, bad taste, delayed shipping and product quality.

3. The retrieval system can help customer support teams
   respond faster by finding similar customer experiences.
""")


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n========================================")
print("EX4 COMPLETED SUCCESSFULLY")
print("========================================")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HDC0422056\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HDC0422056\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HDC0422056\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


NLTK resources downloaded successfully!
Dataset loaded successfully!

Dataset shape:
(568454, 10)

First 5 records:
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                  